In [289]:
import pandas as pd
import regex as re
import io

In [290]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

In [291]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual
df['AF'] = df['INFO'].apply(extract_af)

df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [292]:
type(df['AF'].iloc[0])

str

In [293]:
#get sequnce of individual at each position
og_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/og_ref.csv')

og_ref.shape

(13314, 3)

In [294]:
new_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/1000_genome_new_ref/1000_genome_new_ref_v2.csv')

new_ref.shape

(13314, 3)

In [295]:
df = df[['POS', 'ALT', 'AF']]  

# merged_df = pd.merge(og_ref, new_ref, on='POS', how='left')
merged_df = pd.merge(og_ref, df, on='POS', how='left', suffixes=('_og', '_vcf'))

merged_df.rename(columns={'REF':'ALT_og', 'ALT':'ALT_vcf'}, inplace=True)

merged_df.head(33)

,POS,Type,ALT_og,ALT_vcf,AF
0,1,sequence,G,NaN,NaN
1,2,sequence,C,NaN,NaN
2,3,sequence,T,NaN,NaN
3,4,sequence,G,NaN,NaN
4,5,sequence,A,NaN,NaN
5,6,sequence,C,NaN,NaN
6,7,sequence,A,NaN,NaN
7,8,sequence,C,NaN,NaN
8,9,sequence,G,NaN,NaN
9,10,sequence,C,NaN,NaN


In [296]:
merged_df.shape

(13346, 5)

In [297]:
merged_df[(merged_df['POS'] - 1)!= merged_df.index]

,POS,Type,ALT_og,ALT_vcf,AF
386,386,sequence,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN
388,388,sequence,G,NaN,NaN
389,389,sequence,C,NaN,NaN
390,390,sequence,C,NaN,NaN
...,...,...,...,...,...
13341,13310,sequence,T,NaN,NaN
13342,13311,sequence,C,NaN,NaN
13343,13312,sequence,G,NaN,NaN
13344,13313,sequence,C,NaN,NaN


In [298]:
merged_df.head(389)

,POS,Type,ALT_og,ALT_vcf,AF
0,1,sequence,G,NaN,NaN
1,2,sequence,C,NaN,NaN
2,3,sequence,T,NaN,NaN
3,4,sequence,G,NaN,NaN
4,5,sequence,A,NaN,NaN
...,...,...,...,...,...
384,385,sequence,C,NaN,NaN
385,386,sequence,T,TGGCAA,0.100669
386,386,sequence,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN


In [299]:
df[df['POS'] == 386]

,POS,ALT,AF
13,386,TGGCAA,0.100669
14,386,TGGCC,0.736424


In [300]:
# Combine the ALT_og column and ALT_vcf into ALT_combined, and set AF to 0 where variant did not exist
merged_df['ALT_combined'] = merged_df.apply(lambda x: x['ALT_vcf'] if pd.notna(x['ALT_vcf']) else x['ALT_og'], axis=1)
merged_df['AF'] = merged_df['AF'].fillna(0) 

merged_df.head(32)

,POS,Type,ALT_og,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,0,G
1,2,sequence,C,NaN,0,C
2,3,sequence,T,NaN,0,T
3,4,sequence,G,NaN,0,G
4,5,sequence,A,NaN,0,A
5,6,sequence,C,NaN,0,C
6,7,sequence,A,NaN,0,A
7,8,sequence,C,NaN,0,C
8,9,sequence,G,NaN,0,G
9,10,sequence,C,NaN,0,C


In [301]:
merged_df.drop(columns=['ALT_vcf','Type'], inplace=True)

merged_df.rename(columns={'ALT_og': 'og_ref', 'AF':'AF_old', 'ALT_combined':'ALT_old'}, inplace=True)

merged_df.head(73)

,POS,og_ref,AF_old,ALT_old
0,1,G,0,G
1,2,C,0,C
2,3,T,0,T
3,4,G,0,G
4,5,A,0,A
...,...,...,...,...
68,69,G,0,G
69,70,C,0,C
70,71,C,0,C
71,72,T,0,T


In [302]:
merged_df = pd.merge(merged_df, new_ref, on='POS', how='left')
merged_df.drop(columns=['Unnamed: 0'], inplace=True)
merged_df.rename(columns={'1000_genome_new_ref': 'new_ref'}, inplace=True)

merged_df

,POS,og_ref,AF_old,ALT_old,new_ref
0,1,G,0,G,G
1,2,C,0,C,C
2,3,T,0,T,T
3,4,G,0,G,G
4,5,A,0,A,A
...,...,...,...,...,...
13341,13310,T,0,T,T
13342,13311,C,0,C,C
13343,13312,G,0,G,G
13344,13313,C,0,C,C


In [303]:
filtered_df = merged_df[(merged_df['og_ref'] != merged_df['new_ref']) & (merged_df['AF_old'] != 0)]

filtered_df.shape

(87, 5)

In [304]:
filtered_df

,POS,og_ref,AF_old,ALT_old,new_ref
30,31,T,1.000000,C,C
31,32,C,1.000000,T,T
72,73,C,0.996952,A,A
73,74,A,0.999624,C,C
79,80,A,0.977024,AC,AC
...,...,...,...,...,...
13181,13150,A,0.961884,AC,AC
13250,13219,C,0.944088,CG,CG
13309,13278,G,0.909971,GGC,GGC
13311,13280,G,1.000000,C,C


In [305]:
merged_df['ALT_new'] = merged_df.apply(lambda x: x['og_ref'] if (x['AF_old'] != 0) and (x['og_ref'] != x['new_ref']) else x['ALT_old'], axis=1)

merged_df

,POS,og_ref,AF_old,ALT_old,new_ref,ALT_new
0,1,G,0,G,G,G
1,2,C,0,C,C,C
2,3,T,0,T,T,T
3,4,G,0,G,G,G
4,5,A,0,A,A,A
...,...,...,...,...,...,...
13341,13310,T,0,T,T,T
13342,13311,C,0,C,C,C
13343,13312,G,0,G,G,G
13344,13313,C,0,C,C,C


In [306]:
merged_df.head(32)

,POS,og_ref,AF_old,ALT_old,new_ref,ALT_new
0,1,G,0,G,G,G
1,2,C,0,C,C,C
2,3,T,0,T,T,T
3,4,G,0,G,G,G
4,5,A,0,A,A,A
5,6,C,0,C,C,C
6,7,A,0,A,A,A
7,8,C,0,C,C,C
8,9,G,0,G,G,G
9,10,C,0,C,C,C


In [307]:
merged_df['AF_new'] = merged_df.apply(lambda x: 1 - float(x['AF_old']) if (x['og_ref'] != x['new_ref']) else float(x['AF_old']), axis=1)

merged_df

,POS,og_ref,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,0,G,G,G,0.0
1,2,C,0,C,C,C,0.0
2,3,T,0,T,T,T,0.0
3,4,G,0,G,G,G,0.0
4,5,A,0,A,A,A,0.0
...,...,...,...,...,...,...,...
13341,13310,T,0,T,T,T,0.0
13342,13311,C,0,C,C,C,0.0
13343,13312,G,0,G,G,G,0.0
13344,13313,C,0,C,C,C,0.0


In [308]:
merged_df.iloc[30, :]

POS              31
og_ref            T
AF_old     1.000000
ALT_old           C
new_ref           C
ALT_new           T
AF_new          0.0
Name: 30, dtype: object

In [309]:
merged_df[merged_df['AF_new']!=0]

,POS,og_ref,AF_old,ALT_old,new_ref,ALT_new,AF_new
72,73,C,0.996952,A,A,C,0.003048
73,74,A,0.999624,C,C,A,0.000376
79,80,A,0.977024,AC,AC,A,0.022976
102,103,G,0.987559,G,G,G,0.987559
103,104,C,0,C,0,C,1.000000
...,...,...,...,...,...,...,...
13250,13219,C,0.944088,CG,CG,C,0.055912
13266,13235,G,0.953411,G,G,G,0.953411
13267,13236,C,0,C,0,C,1.000000
13309,13278,G,0.909971,GGC,GGC,G,0.090029


In [312]:
merged_df[merged_df['new_ref']== '0']

,POS,og_ref,AF_old,ALT_old,new_ref,ALT_new,AF_new
103,104,C,0,C,0,C,1.000000
538,538,G,0,G,0,G,1.000000
1674,1672,C,0,C,0,C,1.000000
1675,1673,T,0.318475,T,0,T,0.681525
1676,1674,C,0,C,0,C,1.000000
1677,1675,C,0,C,0,C,1.000000
1678,1676,C,0,C,0,C,1.000000
1759,1757,T,0,T,0,T,1.000000
3156,3153,C,0,C,0,C,1.000000
3415,3412,C,0,C,0,C,1.000000
